In [1]:
# ================================
# 1. Import Libraries
# ================================
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib


# ================================
# 2. Load Model and Data
# ================================
model = joblib.load("../model/churn_model.pkl")
X = pd.read_csv("../model/X_scaled.csv")
y = pd.read_csv("../model/y.csv")
y = y.values.ravel()

print("Model loaded :", type(model).__name__)
print("Data shape   :", X.shape)


# ================================
# 3. Get Churn Probabilities
# ================================
print("\nCalculating churn probabilities...")

churn_proba = model.predict_proba(X)[:, 1]

print(f"Min probability  : {churn_proba.min():.4f}")
print(f"Max probability  : {churn_proba.max():.4f}")
print(f"Mean probability : {churn_proba.mean():.4f}")


# ================================
# 4. Assign Risk Segments
# ================================
def assign_segment(prob):
    if prob >= 0.7:
        return "High Risk"
    elif prob >= 0.4:
        return "Medium Risk"
    else:
        return "Low Risk"

segments = [assign_segment(p) for p in churn_proba]


# ================================
# 5. Build Segmented DataFrame
# ================================
df_segmented = X.copy()
df_segmented['Churn_Probability'] = churn_proba
df_segmented['Risk_Segment']      = segments
df_segmented['Actual_Churn']      = y

# ================================
# 6. Segment Summary
# ================================
print("\n--- Segment Distribution ---")
segment_counts = df_segmented['Risk_Segment'].value_counts()
print(segment_counts)

print("\n--- Segment Summary Table ---")
summary = df_segmented.groupby('Risk_Segment').agg(
    Customer_Count   = ('Churn_Probability', 'count'),
    Avg_Churn_Prob   = ('Churn_Probability', 'mean'),
    Min_Prob         = ('Churn_Probability', 'min'),
    Max_Prob         = ('Churn_Probability', 'max'),
    Actual_Churners  = ('Actual_Churn',      'sum')
).round(4)

summary['Churn_Rate_%'] = (
    summary['Actual_Churners'] / summary['Customer_Count'] * 100
).round(2)

print(summary.to_string())


# ================================
# 7. Segment Distribution Plot
# ================================
print("\nGenerating segment distribution plot...")

colors = {'High Risk': '#E24B4A', 'Medium Risk': '#EF9F27', 'Low Risk': '#1D9E75'}
segment_order = ['High Risk', 'Medium Risk', 'Low Risk']
counts = [segment_counts.get(s, 0) for s in segment_order]
bar_colors = [colors[s] for s in segment_order]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar(segment_order, counts, color=bar_colors, edgecolor='white', linewidth=0.5)
axes[0].set_title('Customer Count by Risk Segment', fontsize=13)
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(counts):
    axes[0].text(i, v + 30, str(v), ha='center', fontsize=11, fontweight='bold')

# Pie chart
axes[1].pie(
    counts,
    labels=segment_order,
    colors=bar_colors,
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 11}
)
axes[1].set_title('Risk Segment Distribution', fontsize=13)

plt.suptitle('Churn Risk Segmentation', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("../model/segment_distribution.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved → segment_distribution.png")


# ================================
# 8. Probability Distribution Plot
# ================================
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    df_segmented[df_segmented['Risk_Segment'] == 'Low Risk']['Churn_Probability'],
    bins=30, alpha=0.7, color='#1D9E75', label='Low Risk'
)
ax.hist(
    df_segmented[df_segmented['Risk_Segment'] == 'Medium Risk']['Churn_Probability'],
    bins=30, alpha=0.7, color='#EF9F27', label='Medium Risk'
)
ax.hist(
    df_segmented[df_segmented['Risk_Segment'] == 'High Risk']['Churn_Probability'],
    bins=30, alpha=0.7, color='#E24B4A', label='High Risk'
)

ax.axvline(x=0.4, color='gray', linestyle='--', linewidth=1, label='Threshold 0.4')
ax.axvline(x=0.7, color='black', linestyle='--', linewidth=1, label='Threshold 0.7')
ax.set_title('Churn Probability Distribution by Segment', fontsize=13)
ax.set_xlabel('Churn Probability')
ax.set_ylabel('Number of Customers')
ax.legend()
plt.tight_layout()
plt.savefig("../model/segment_probability_dist.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved → segment_probability_dist.png")


# ================================
# 9. Key Business Metrics per Segment
# ================================
print("\n--- Key Business Metrics per Segment ---")

business_summary = df_segmented.groupby('Risk_Segment').agg(
    Customers        = ('Churn_Probability', 'count'),
    Avg_Monthly_Charge = ('Monthly Charges', 'mean'),
    Avg_Tenure       = ('Tenure Months',    'mean'),
).round(2)

print(business_summary.to_string())


# ================================
# 10. Save Segmented Data
# ================================
df_segmented.to_csv("../model/segmented_customers.csv", index=False)
print("\nSaved → segmented_customers.csv")

print("\nSegmentation Complete!")
print("Files saved:")
print("  segment_distribution.png")
print("  segment_probability_dist.png")
print("  segmented_customers.csv")

print(f"\nQuick Summary:")
print(f"  High Risk customers   : {segment_counts.get('High Risk', 0)}")
print(f"  Medium Risk customers : {segment_counts.get('Medium Risk', 0)}")
print(f"  Low Risk customers    : {segment_counts.get('Low Risk', 0)}")

Model loaded : XGBClassifier
Data shape   : (7043, 30)

Calculating churn probabilities...
Min probability  : 0.0001
Max probability  : 0.9946
Mean probability : 0.4413

--- Segment Distribution ---
Risk_Segment
Low Risk       3590
High Risk      2487
Medium Risk     966
Name: count, dtype: int64

--- Segment Summary Table ---
              Customer_Count  Avg_Churn_Prob  Min_Prob  Max_Prob  Actual_Churners  Churn_Rate_%
Risk_Segment                                                                                   
High Risk               2487          0.8803    0.7002    0.9946             1600         64.33
Low Risk                3590          0.1069    0.0001    0.3999               55          1.53
Medium Risk              966          0.5538    0.4001    0.6994              214         22.15

Generating segment distribution plot...
Saved → segment_distribution.png
Saved → segment_probability_dist.png

--- Key Business Metrics per Segment ---
              Customers  Avg_Monthly_C